In [ ]:
"""
Standalone cell for computing unconditional differential entropy with KDE.

This estimates H(P_t) as:
    H(P_t) ~= - mean_test log p_KDE(test)

KDE is fit on train labels and bandwidth is selected by validation likelihood.
For positive skewed features, KDE is applied in log-space with Jacobian correction.
"""

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import KernelDensity


PKL_BASE = Path(r"./Data/prosodic_features_splits_CLEANED")

TARGET_SPLITS = [
    "full",
    "healthy",
    "depressed",
    "psychosis",
    "remission_Y",
    "remission_N",
]

test_file_by_split = {
    "full": "test.pkl",
    "healthy": "test_healthy.pkl",
    "depressed": "test_depression.pkl",
    "psychosis": "test_psychosis.pkl",
    "remission_Y": "test_psychosis_remission_Y.pkl",
    "remission_N": "test_psychosis_remission_N.pkl",
}

features = [
    "absolute_prominence",
    "relative_prominence",
    "duration",
    "loudness",
    "pause",
    "pitch",
]

# KDE handling:
# raw          = KDE directly on scalar values
# log_positive = log-transform positive scalar values, KDE there, then Jacobian correction
# multivariate = KDE on vector-valued pitch labels
kde_type_by_feature = {
    "absolute_prominence": "log_positive",
    "relative_prominence": "raw",
    "duration": "raw",
    "loudness": "raw",
    "pause": "log_positive",
    "pitch": "multivariate",
}

BANDWIDTH_GRID = np.array([
    0.03, 0.05, 0.075, 0.1, 0.15,
    0.2, 0.3, 0.5, 0.75, 1.0,
])

RANDOM_SEED = 12345

# Reduce these if KDE is too slow
MAX_TRAIN_KDE = 50000
MAX_VAL_KDE = 20000
MAX_TEST_EVAL = None  # e.g. 50000 if needed


# helpers

def load_labels_auto(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    rows = []

    for seq in data["labels"]:
        for y in seq:
            arr = np.asarray(y, dtype=float)

            if arr.ndim == 0:
                rows.append(float(arr))
            elif arr.ndim == 1:
                rows.append(arr)
            else:
                raise ValueError(f"Unexpected label shape {arr.shape} in {pkl_path}")

    x = np.asarray(rows, dtype=float)

    if x.ndim == 1:
        return x[np.isfinite(x)]

    if x.ndim == 2:
        return x[np.isfinite(x).all(axis=1)]

    raise ValueError(f"Unexpected final shape {x.shape}")


def sample_rows(x, max_n, rng):
    x = np.asarray(x)

    if max_n is None or len(x) <= max_n:
        return x

    idx = rng.choice(len(x), size=max_n, replace=False)
    return x[idx]


def transform_for_kde(x, kde_type, fit_stats=None, eps=1e-6):
    """
    Returns:
        z: transformed standardized values
        log_jacobian: correction so log p_x(x) = log p_z(z) + log_jacobian
        fit_stats: mean/std from train transform
    """
    x = np.asarray(x, dtype=float)

    if kde_type == "raw":
        x = x.reshape(-1, 1)

        if fit_stats is None:
            mu = np.mean(x, axis=0)
            sd = np.maximum(np.std(x, axis=0), eps)
            fit_stats = {"mu": mu, "sd": sd}

        z = (x - fit_stats["mu"]) / fit_stats["sd"]
        log_jacobian = -np.sum(np.log(fit_stats["sd"]))

        return z, log_jacobian, fit_stats

    if kde_type == "log_positive":
        x = np.clip(x.reshape(-1, 1), eps, None)
        lx = np.log(x)

        if fit_stats is None:
            mu = np.mean(lx, axis=0)
            sd = np.maximum(np.std(lx, axis=0), eps)
            fit_stats = {"mu": mu, "sd": sd}

        z = (lx - fit_stats["mu"]) / fit_stats["sd"]

        # z = (log(x) - mu) / sd
        # dz/dx = 1 / (sd * x)
        log_jacobian = -np.log(fit_stats["sd"][0]) - np.log(x[:, 0])

        return z, log_jacobian, fit_stats

    if kde_type == "multivariate":
        if x.ndim != 2:
            raise ValueError("multivariate KDE expects shape (n_samples, n_dims)")

        if fit_stats is None:
            mu = np.mean(x, axis=0)
            sd = np.maximum(np.std(x, axis=0), eps)
            fit_stats = {"mu": mu, "sd": sd}

        z = (x - fit_stats["mu"]) / fit_stats["sd"]
        log_jacobian = -np.sum(np.log(fit_stats["sd"]))

        return z, log_jacobian, fit_stats

    raise ValueError(f"Unknown kde_type: {kde_type}")


def score_kde_in_batches(kde, z, batch_size=10000):
    out = []

    for i in range(0, len(z), batch_size):
        out.append(kde.score_samples(z[i:i + batch_size]))

    return np.concatenate(out)


def estimate_unconditional_nll_kde(
    train_x,
    val_x,
    test_x,
    kde_type,
    bandwidth_grid=BANDWIDTH_GRID,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED)

    train_x = sample_rows(train_x, MAX_TRAIN_KDE, rng)
    val_x = sample_rows(val_x, MAX_VAL_KDE, rng)
    test_x = sample_rows(test_x, MAX_TEST_EVAL, rng)

    z_train, _, fit_stats = transform_for_kde(train_x, kde_type, fit_stats=None)
    z_val, log_jac_val, _ = transform_for_kde(val_x, kde_type, fit_stats=fit_stats)
    z_test, log_jac_test, _ = transform_for_kde(test_x, kde_type, fit_stats=fit_stats)

    best_bw = None
    best_val_ll = -np.inf
    best_kde = None

    for bw in bandwidth_grid:
        kde = KernelDensity(kernel="gaussian", bandwidth=float(bw))
        kde.fit(z_train)

        logpz_val = score_kde_in_batches(kde, z_val)
        logpx_val = logpz_val + log_jac_val

        val_ll = float(np.mean(logpx_val))

        if val_ll > best_val_ll:
            best_val_ll = val_ll
            best_bw = float(bw)
            best_kde = kde

    logpz_test = score_kde_in_batches(best_kde, z_test)
    logpx_test = logpz_test + log_jac_test

    h_uncond = float(-np.mean(logpx_test))

    return {
        "h_uncond": h_uncond,
        "bandwidth": best_bw,
        "val_loglik": best_val_ll,
        "n_train": len(train_x),
        "n_val": len(val_x),
        "n_test": len(test_x),
    }


# KDE entropy estimation!

rng = np.random.default_rng(RANDOM_SEED)

unconditional_nll_by_split = {}
kde_info_by_split = {}

for split in TARGET_SPLITS:
    print("\n==============================")
    print(f"TARGET SPLIT: {split}")
    print("==============================")

    unconditional_nll_by_split[split] = {}
    kde_info_by_split[split] = {}

    for feature in features:
        train_path = PKL_BASE / feature / "train.pkl"
        val_path = PKL_BASE / feature / "val.pkl"
        test_path = PKL_BASE / feature / test_file_by_split[split]

        train_x = load_labels_auto(train_path)
        val_x = load_labels_auto(val_path)
        test_x = load_labels_auto(test_path)

        kde_type = kde_type_by_feature[feature]

        result = estimate_unconditional_nll_kde(
            train_x=train_x,
            val_x=val_x,
            test_x=test_x,
            kde_type=kde_type,
            rng=rng,
        )

        h_uncond = result["h_uncond"]

        unconditional_nll_by_split[split][feature] = h_uncond
        kde_info_by_split[split][feature] = result

        print(f"\n=== {feature} ===")
        print(f"KDE type:                {kde_type}")
        print(f"train shape:             {train_x.shape}")
        print(f"val shape:               {val_x.shape}")
        print(f"test shape:              {test_x.shape}")
        print(f"bandwidth:               {result['bandwidth']}")
        print(f"val loglik:              {result['val_loglik']:.6f}")
        print(f"H_uncond / KDE NLL:      {h_uncond:.6f}")

        if train_x.ndim == 1:
            print(
                f"train mean={np.mean(train_x):.4f}, std={np.std(train_x):.4f} | "
                f"test mean={np.mean(test_x):.4f}, std={np.std(test_x):.4f}"
            )
        else:
            print(f"train mean={np.mean(train_x, axis=0)}")
            print(f"train std ={np.std(train_x, axis=0)}")
            print(f"test mean ={np.mean(test_x, axis=0)}")
            print(f"test std  ={np.std(test_x, axis=0)}")


summary = pd.DataFrame(unconditional_nll_by_split).T[features]

print("\n\n--- KDE unconditional entropy / marginal NLL summary ---")
display(summary)

print("\nCopyable:")
print(summary.round(6).to_string())

print("\nBandwidths:")
bandwidth_summary = pd.DataFrame({
    split: {
        feature: kde_info_by_split[split][feature]["bandwidth"]
        for feature in features
    }
    for split in TARGET_SPLITS
}).T[features]

display(bandwidth_summary)


TARGET SPLIT: full

=== absolute_prominence ===
family:                  gamma
train shape:             (458082,)
test shape:              (67338,)
H_uncond / marginal NLL: -1.218600
train mean=0.1306, std=0.1299 | test mean=0.1304, std=0.1078

=== relative_prominence ===
family:                  gaussian
train shape:             (458082,)
test shape:              (67338,)
H_uncond / marginal NLL: -0.615530
train mean=0.0039, std=0.1484 | test mean=0.0043, std=0.1282

=== duration ===
family:                  gaussian
train shape:             (456337,)
test shape:              (67178,)
H_uncond / marginal NLL: -0.281901
train mean=0.2315, std=0.1819 | test mean=0.2354, std=0.1825

=== loudness ===
family:                  gaussian
train shape:             (458024,)
test shape:              (67333,)
H_uncond / marginal NLL: 1.418327
train mean=-0.0003, std=0.9979 | test mean=-0.0000, std=0.9994

=== pause ===
family:                  gamma
train shape:             (457610,)
test shape:

,absolute_prominence,relative_prominence,duration,loudness,pause,pitch
full,-1.218600,-0.615530,-0.281901,1.418327,-4.729240,5.267505
healthy,-1.275367,-0.744056,-0.334114,1.418030,-4.709973,5.248890
depressed,-1.307019,-0.651445,-0.371043,1.418531,-4.740630,5.314385
psychosis,-1.094857,-0.406098,-0.158932,1.418458,-4.740308,5.259975



Copyable:
           absolute_prominence  relative_prominence  duration  loudness     pause     pitch
full                 -1.218600            -0.615530 -0.281901  1.418327 -4.729240  5.267505
healthy              -1.275367            -0.744056 -0.334114  1.418030 -4.709973  5.248890
depressed            -1.307019            -0.651445 -0.371043  1.418531 -4.740630  5.314385
psychosis            -1.094857            -0.406098 -0.158932  1.418458 -4.740308  5.259975
